# Wanderbricks — Test the Serving Endpoint

Call the deployed `wanderbricks-weather-serve` endpoint with a realistic payload built from the actual feature table. Measures round-trip latency (includes network — the endpoint UI's Metrics tab shows the server-side p50/p95/p99 latency).

**Prereqs:** the DLT pipeline has run, a model version is registered, and the serving endpoint is deployed (rename the `.example` resource and `databricks bundle deploy`).

**Dev-mode name:** resources are prefixed `dev_<user>_` in the dev
target — the endpoint is `dev_iraonfridays_wanderbricks-weather-serve`
here; in prod it is `wanderbricks-weather-serve`.

In [ ]:
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema", "iraonfridays")
dbutils.widgets.text("endpoint", "dev_iraonfridays_wanderbricks-weather-serve")
CATALOG = dbutils.widgets.get("catalog").strip()
SCHEMA = dbutils.widgets.get("schema").strip()
ENDPOINT = dbutils.widgets.get("endpoint").strip()
print(f"endpoint: {ENDPOINT}  tables: {CATALOG}.{SCHEMA}")

## Build a realistic request payload

In [ ]:
import pyspark.sql.functions as F

# One real feature row per station (drop non-model columns)
feats = (
    spark.table(f"{CATALOG}.{SCHEMA}.weather_features")
    .drop("station", "date", "temp_c")
)
sample = feats.dropna().limit(1).toPandas().to_dict(orient="records")
print(sample)
assert sample, "weather_features is empty - run the DLT pipeline first"

## Call the endpoint and time it

In [ ]:
import requests, time, json

ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
host = ctx.apiUrl().get()
token = ctx.apiToken().get()

payload = {"dataframe_records": sample}
t0 = time.perf_counter()
resp = requests.post(
    f"{host}/serving-endpoints/{ENDPOINT}/invocations",
    headers={"Authorization": f"Bearer {token}"},
    json=payload,
    timeout=60,
)
elapsed = time.perf_counter() - t0
print(f"HTTP {resp.status_code} · round-trip {elapsed:.4f}s")
if resp.status_code == 200:
    print(json.dumps(resp.json(), indent=2))
else:
    print(resp.text)

## Reading the results

- **Round-trip** here includes network + endpoint dispatch; the server-side number is in the endpoint's **Metrics tab** (p50/p95/p99 latency, requests/sec).
- **First call after idle** may be slower — `scale_to_zero_enabled: true` means cold starts; keep it on for cost, off for guaranteed latency.
- The response should match the model signature (a `predictions` array of temperatures in °C).

Compare with the batch path: `SELECT * FROM <schema>.scoring_metrics ORDER BY run_date` shows `inference_seconds` from `06_score.py`.